# Project 60 — Local Browser Task Agent

**Navigate simple web tasks in a controlled notebook prototype — with simulated browser state.**

| Component | Choice |
|-----------|--------|
| LLM runtime | Ollama (qwen3:8b) |
| Framework | LangChain |
| Browser | Simulated DOM state |
| Interface | Jupyter notebook |

> **Note:** This project simulates browser interaction using a mock DOM model.
> In production, you would use tools like Playwright or Selenium with a real browser.

## 1 · What You Will Learn

1. How to model a **simulated web page** as structured state for an LLM agent
2. How to let an LLM **plan and execute** multi-step web interactions
3. How to implement **action primitives** (click, type, scroll, read) for web navigation
4. How to build a **task decomposition** pipeline for complex web tasks
5. How to evaluate whether an agent **completed** a web task successfully

## 2 · Why This Project Matters

Browser automation is one of the most requested agentic AI applications — from
filling forms to extracting data from websites. Understanding how to model web
state, define actions, and verify task completion is foundational for building
real browser agents. We start with simulation to learn the patterns safely.

## 3 · Environment Setup

**Prerequisites:**
- Ollama running with `qwen3:8b`

In [ ]:
# !pip install -q langchain langchain-ollama

## 4 · Imports and Configuration

In [ ]:
import json
from typing import Optional

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = 'qwen3:8b'
llm = ChatOllama(model=MODEL, temperature=0.1)
print(f'LLM ready: {MODEL}')

## 5 · Build the Simulated Browser

We model web pages as structured data: each page has a URL, visible elements,
and interactive components. The agent interacts through action primitives.

In [ ]:
class SimulatedBrowser:
    """A mock browser for teaching agent web interaction patterns."""

    def __init__(self):
        self.current_page = None
        self.history = []
        self.form_data = {}
        self.action_log = []

        # Define available pages
        self.pages = {
            'https://shop.example.com': {
                'title': 'Example Shop - Home',
                'elements': [
                    {'id': 'search-box', 'type': 'input', 'placeholder': 'Search products...'},
                    {'id': 'search-btn', 'type': 'button', 'text': 'Search'},
                    {'id': 'cat-electronics', 'type': 'link', 'text': 'Electronics', 'href': 'https://shop.example.com/electronics'},
                    {'id': 'cat-books', 'type': 'link', 'text': 'Books', 'href': 'https://shop.example.com/books'},
                    {'id': 'cart-icon', 'type': 'link', 'text': 'Cart (0)', 'href': 'https://shop.example.com/cart'},
                    {'id': 'login-btn', 'type': 'button', 'text': 'Sign In'},
                ],
                'content': 'Welcome to Example Shop! Browse our categories or search for products.',
            },
            'https://shop.example.com/electronics': {
                'title': 'Electronics - Example Shop',
                'elements': [
                    {'id': 'prod-1', 'type': 'link', 'text': 'Wireless Mouse - $24.99', 'href': 'https://shop.example.com/product/mouse'},
                    {'id': 'prod-2', 'type': 'link', 'text': 'USB-C Hub - $34.99', 'href': 'https://shop.example.com/product/hub'},
                    {'id': 'prod-3', 'type': 'link', 'text': 'Mechanical Keyboard - $79.99', 'href': 'https://shop.example.com/product/keyboard'},
                    {'id': 'filter-price', 'type': 'select', 'options': ['All Prices', 'Under $25', '$25-$50', 'Over $50']},
                    {'id': 'back-home', 'type': 'link', 'text': 'Back to Home', 'href': 'https://shop.example.com'},
                ],
                'content': 'Electronics - 3 products found.',
            },
            'https://shop.example.com/product/mouse': {
                'title': 'Wireless Mouse - Example Shop',
                'elements': [
                    {'id': 'add-cart', 'type': 'button', 'text': 'Add to Cart'},
                    {'id': 'qty', 'type': 'input', 'value': '1', 'placeholder': 'Quantity'},
                    {'id': 'back-cat', 'type': 'link', 'text': 'Back to Electronics', 'href': 'https://shop.example.com/electronics'},
                ],
                'content': 'Wireless Mouse\nPrice: $24.99\nIn Stock: Yes\nRating: 4.5/5 (234 reviews)\nFeatures: Bluetooth, USB-C charging, 6 buttons',
            },
            'https://shop.example.com/cart': {
                'title': 'Shopping Cart - Example Shop',
                'elements': [
                    {'id': 'checkout-btn', 'type': 'button', 'text': 'Proceed to Checkout'},
                    {'id': 'continue-btn', 'type': 'link', 'text': 'Continue Shopping', 'href': 'https://shop.example.com'},
                ],
                'content': 'Your cart is empty.',
            },
            'https://shop.example.com/login': {
                'title': 'Sign In - Example Shop',
                'elements': [
                    {'id': 'email-input', 'type': 'input', 'placeholder': 'Email address'},
                    {'id': 'pass-input', 'type': 'input', 'placeholder': 'Password'},
                    {'id': 'login-submit', 'type': 'button', 'text': 'Sign In'},
                    {'id': 'forgot-pass', 'type': 'link', 'text': 'Forgot Password?', 'href': '#'},
                ],
                'content': 'Sign in to your account.',
            },
        }

    def navigate(self, url: str) -> str:
        """Go to a URL."""
        if url in self.pages:
            self.current_page = url
            self.history.append(url)
            self.action_log.append(f'NAVIGATE: {url}')
            return self.get_page_state()
        return f'Page not found: {url}'

    def click(self, element_id: str) -> str:
        """Click an element on the current page."""
        if not self.current_page:
            return 'No page loaded'
        page = self.pages[self.current_page]
        for elem in page['elements']:
            if elem['id'] == element_id:
                self.action_log.append(f'CLICK: {element_id} ({elem.get("text", "")})')
                # Handle navigation links
                if elem['type'] == 'link' and 'href' in elem:
                    return self.navigate(elem['href'])
                # Handle buttons
                if elem['type'] == 'button':
                    if element_id == 'add-cart':
                        return 'Item added to cart! Cart now has 1 item.'
                    if element_id == 'login-btn':
                        return self.navigate('https://shop.example.com/login')
                    if element_id == 'search-btn':
                        query = self.form_data.get('search-box', '')
                        return f'Search results for "{query}": Showing relevant products...'
                    return f'Clicked: {elem.get("text", element_id)}'
                return f'Interacted with: {element_id}'
        return f'Element not found: {element_id}'

    def type_text(self, element_id: str, text: str) -> str:
        """Type text into an input field."""
        if not self.current_page:
            return 'No page loaded'
        page = self.pages[self.current_page]
        for elem in page['elements']:
            if elem['id'] == element_id and elem['type'] == 'input':
                self.form_data[element_id] = text
                self.action_log.append(f'TYPE: "{text}" into {element_id}')
                return f'Typed "{text}" into {element_id}'
        return f'Input field not found: {element_id}'

    def read_page(self) -> str:
        """Read the current page content."""
        if not self.current_page:
            return 'No page loaded'
        return self.get_page_state()

    def get_page_state(self) -> str:
        """Get a text description of the current page state."""
        if not self.current_page:
            return 'No page loaded'
        page = self.pages[self.current_page]
        lines = [
            f'URL: {self.current_page}',
            f'Title: {page["title"]}',
            f'Content: {page["content"]}',
            'Interactive elements:',
        ]
        for elem in page['elements']:
            desc = f'  [{elem["id"]}] {elem["type"].upper()}'
            if 'text' in elem:
                desc += f': "{elem["text"]}"'
            if 'placeholder' in elem:
                desc += f' (placeholder: {elem["placeholder"]})'
            lines.append(desc)
        return '\n'.join(lines)


# Initialize and test
browser = SimulatedBrowser()
print(browser.navigate('https://shop.example.com'))

## 6 · Build the Task Planning Agent

The agent takes a high-level task description, decomposes it into browser actions,
and executes them step by step.

In [ ]:
def plan_browser_task(task: str, browser_state: str) -> list:
    """Use LLM to decompose a task into browser actions."""
    prompt = ChatPromptTemplate.from_messages([
        ('system',
         'You are a browser automation agent. Given a task and the current page state, '
         'produce a step-by-step plan.\n\n'
         'Available actions:\n'
         '- NAVIGATE url\n'
         '- CLICK element_id\n'
         '- TYPE element_id "text"\n'
         '- READ (read current page)\n\n'
         'Respond with ONLY a JSON list of action objects (no markdown):\n'
         '[{{"action": "CLICK", "target": "element_id", "reason": "why"}}, ...]'),
        ('human', 'Task: {task}\n\nCurrent page state:\n{state}'),
    ])
    chain = prompt | llm | StrOutputParser()
    raw = chain.invoke({'task': task, 'state': browser_state})

    try:
        start = raw.find('[')
        end = raw.rfind(']') + 1
        if start >= 0 and end > start:
            return json.loads(raw[start:end])
    except json.JSONDecodeError:
        pass
    return [{'action': 'READ', 'target': '', 'reason': 'Could not parse plan'}]


def execute_browser_task(task: str, verbose: bool = True) -> dict:
    """Plan and execute a browser task."""
    # Reset browser
    browser_local = SimulatedBrowser()
    browser_local.navigate('https://shop.example.com')

    if verbose:
        print(f'\nTASK: {task}')
        print('=' * 60)

    # Plan
    state = browser_local.get_page_state()
    plan = plan_browser_task(task, state)

    if verbose:
        print(f'\nPlan ({len(plan)} steps):')
        for i, step in enumerate(plan, 1):
            print(f'  {i}. {step.get("action", "?")}: {step.get("target", "")} - {step.get("reason", "")}')

    # Execute
    results = []
    for i, step in enumerate(plan, 1):
        action = step.get('action', '').upper()
        target = step.get('target', '')
        text = step.get('text', '')

        if action == 'NAVIGATE':
            result = browser_local.navigate(target)
        elif action == 'CLICK':
            result = browser_local.click(target)
        elif action == 'TYPE':
            result = browser_local.type_text(target, text)
        elif action == 'READ':
            result = browser_local.read_page()
        else:
            result = f'Unknown action: {action}'

        results.append({'step': i, 'action': action, 'target': target, 'result': result[:200]})

        if verbose:
            print(f'\n  Step {i} [{action}]: {result[:150]}')

    # Verify completion
    verify_prompt = ChatPromptTemplate.from_messages([
        ('system',
         'You are a QA agent. Given a task and the execution log, determine if the '
         'task was completed successfully. Respond with: '
         '"PASS: [reason]" or "FAIL: [reason]"'),
        ('human', 'Task: {task}\nExecution log: {log}'),
    ])
    verify_chain = verify_prompt | llm | StrOutputParser()
    verification = verify_chain.invoke({'task': task, 'log': json.dumps(results, indent=1)})

    if verbose:
        print(f'\nVerification: {verification}')

    return {
        'task': task,
        'plan': plan,
        'results': results,
        'action_log': browser_local.action_log,
        'verification': verification,
    }


print('Browser task agent ready.')

## 7 · Run Browser Tasks

In [ ]:
t1 = execute_browser_task('Browse the electronics category and find the wireless mouse product')

In [ ]:
t2 = execute_browser_task('Go to the electronics section and check the price of the USB-C hub')

In [ ]:
t3 = execute_browser_task('Navigate to the login page')

In [ ]:
t4 = execute_browser_task('Search for headphones on the website')

## 8 · Task Completion Summary

In [ ]:
print('TASK COMPLETION SUMMARY')
print('=' * 60)
for t in [t1, t2, t3, t4]:
    status = 'PASS' if 'PASS' in t.get('verification', '') else 'NEEDS REVIEW'
    print(f'\n  Task: {t["task"][:50]}')
    print(f'  Steps: {len(t["plan"])}')
    print(f'  Status: {status}')
    print(f'  Actions: {t["action_log"]}')

## 9 · Save Results

In [ ]:
output = [t1, t2, t3, t4]
with open('browser_agent_results.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f'Saved {len(output)} task results')

## 10 · Limitations

- **Simulated browser** — no real web pages or dynamic content
- **Fixed page set** — only a small set of mock pages available
- **No visual understanding** — the agent works from text descriptions, not screenshots
- **No JavaScript** — no dynamic page updates or AJAX
- **Plan parsing** can fail with small local models

## 11 · How to Improve

1. **Add Playwright/Selenium**: connect to a real browser for live web interaction
2. **Screenshot understanding**: use a VLM to understand page screenshots
3. **DOM extraction**: parse real HTML into the structured format
4. **Error recovery**: retry failed actions with alternative approaches
5. **Session management**: handle cookies, login state, and multi-tab navigation

## 12 · Mini Challenge

1. Add a checkout page and implement a 'buy the wireless mouse' task
2. Add form validation (e.g., email format check on the login page)
3. Implement a 'compare two products' task that reads multiple product pages

## 13 · Key Takeaways

| Concept | What You Practiced |
|---------|-------------------|
| Browser simulation | Model web pages as structured state |
| Action primitives | Navigate, click, type, read |
| Task decomposition | LLM breaks high-level tasks into steps |
| Verification | Automated pass/fail checking |
| Local-first | Ollama + Python simulation — no real browser needed |